In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Pengundur-belakang pengoperasi (OBP) untuk penganggaran nilai jangkaan

*Anggaran penggunaan: 16 minit pada pemproses Eagle r3 (NOTA: Ini hanyalah anggaran. Masa larian anda mungkin berbeza.)*

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## Latar Belakang
Pengundur-belakang pengoperasi ialah teknik yang melibatkan penyerapan operasi dari hujung Circuit kuantum ke dalam observable yang diukur, yang secara amnya mengurangkan kedalaman Circuit dengan kos tambahan terma dalam observable. Matlamatnya ialah untuk mengundurkan sebanyak mungkin Circuit tanpa membiarkan observable membesar terlalu besar. Pelaksanaan berasaskan Qiskit tersedia dalam addon OBP Qiskit, butiran lanjut boleh didapati dalam [dokumentasi](/guides/qiskit-addons-obp) yang berkaitan dengan [contoh mudah](/guides/qiskit-addons-obp-get-started) untuk memulakan.

Pertimbangkan contoh Circuit di mana observable $O = \sum_P c_P P$ hendak diukur, di mana $P$ ialah Pauli dan $c_P$ ialah pekali. Mari kita tandakan Circuit tersebut sebagai satu kesatuan $U$ yang boleh dibahagikan secara logik kepada $U = U_C U_Q$ seperti yang ditunjukkan dalam rajah di bawah.

![Circuit diagram showing Uq followed by Uc](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

Pengundur-belakang pengoperasi menyerap kesatuan $U_C$ ke dalam observable dengan mengevolusinya sebagai $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$. Dengan kata lain, sebahagian pengiraan dilakukan secara klasik melalui evolusi observable daripada $O$ kepada $O'$. Masalah asal kini boleh diformulasi semula sebagai pengukuran observable $O'$ untuk Circuit kedalaman rendah baru yang kesatuannya ialah $U_Q$.

Kesatuan $U_C$ diwakili sebagai beberapa hirisan $U_C = U_S U_{S-1}...U_2U_1$. Terdapat pelbagai cara untuk mentakrifkan hirisan. Sebagai contoh, dalam Circuit contoh di atas, setiap lapisan $R_{zz}$ dan setiap lapisan gate $R_x$ boleh dianggap sebagai hirisan individu. Pengundur-belakang melibatkan pengiraan $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$ secara klasik. Setiap hirisan $U_s$ boleh diwakili sebagai $U_s = exp(\frac{-i\theta_s P_s}{2})$, di mana $P_s$ ialah Pauli $n$-Qubit dan $\theta_s$ ialah skalar. Mudah untuk disahkan bahawa

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$
$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

Dalam contoh di atas, jika ${P,P_s} = 0$, maka kita perlu melaksanakan dua Circuit kuantum, dan bukannya satu, untuk mengira nilai jangkaan. Oleh itu, pengundur-belakang mungkin meningkatkan bilangan terma dalam observable, yang membawa kepada bilangan pelaksanaan Circuit yang lebih tinggi. Satu cara untuk membenarkan pengundur-belakang yang lebih dalam ke dalam Circuit, sambil mencegah pengoperasi daripada membesar terlalu besar, ialah dengan memangkas terma yang mempunyai pekali kecil, dan bukannya menambahkannya ke dalam pengoperasi. Sebagai contoh, dalam contoh di atas, seseorang mungkin memilih untuk memangkas terma yang melibatkan $P_sP$ dengan syarat $\theta_s$ cukup kecil. Pemangkasan terma boleh menghasilkan lebih sedikit Circuit kuantum untuk dilaksanakan, tetapi melakukan demikian mengakibatkan sedikit ralat dalam pengiraan nilai jangkaan akhir yang berkadar dengan magnitud pekali terma yang dipangkas.

Tutorial ini melaksanakan [corak Qiskit](/guides/intro-to-patterns) untuk mensimulasi dinamik kuantum rantai spin Heisenberg menggunakan <a href="https://github.com/Qiskit/qiskit-addon-obp">qiskit-addon-obp</a>.
## Keperluan
Sebelum memulakan tutorial ini, pastikan anda telah memasang perkara berikut:

- Qiskit SDK v1.2 atau lebih baru (`pip install qiskit`)
- Qiskit Runtime v0.28 atau lebih baru (`pip install qiskit-ibm-runtime`)
- OBP Qiskit addon (`pip install qiskit-addon-obp`)
- Qiskit addon utils (`pip install qiskit-addon-utils`)
## Persediaan

In [2]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

## Bahagian I: Rantai spin Heisenberg skala kecil
### Langkah 1: Petakan input klasik kepada masalah kuantum
#### Petakan evolusi masa model kuantum Heisenberg kepada eksperimen kuantum.
Pakej [qiskit_addon_utils](https://github.com/Qiskit/qiskit-addon-utils) menyediakan beberapa fungsi yang boleh digunakan semula untuk pelbagai tujuan.

Modul [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) menyediakan fungsi untuk menjana Hamiltonian seperti Heisenberg pada graf sambungan yang diberikan.
Graf ini boleh berupa sama ada [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html) atau [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap) menjadikannya mudah digunakan dalam aliran kerja berpusatkan Qiskit.

Berikut, kita menjana `CouplingMap` rantai linear 10 Qubit.

In [3]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/de8ce35e-a2c5-474f-adb9-88c9afb2bd76-0.avif)

Seterusnya, kita menjana pengoperasi Pauli yang memodelkan Hamiltonian Heisenberg XYZ.

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

Di mana $G(V,E)$ ialah graf peta gandingan yang diberikan.

In [4]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", fold=-1)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum hardware execution

#### Create circuit slices to backpropagate

The `backpropagate` function backpropagates entire circuit slices at a time. Therefore, the choice of slicing can have an impact on how well backpropagation performs for a given problem. Here, we will group gates of the same type into slices using the [`slice_by_depth`](/docs/api/qiskit-addon-utils/slicing#slice_by_depth) function.

For a more detailed discussion on circuit slicing, check out this [how-to guide](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) of the [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils) package.

In [5]:
slices = slice_by_depth(circuit, max_slice_depth=1)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


Daripada pengoperasi Qubit, kita boleh menjana Circuit kuantum yang memodelkan evolusi masanya.
Sekali lagi, modul [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) datang membantu dengan fungsi yang berguna untuk melakukan tepat itu:

In [6]:
op_budget = OperatorBudget(max_qwc_groups=8)

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/1d68f197-ffa4-49de-9fe8-243b1facbd00-0.avif)

### Langkah 2: Optimumkan masalah untuk pelaksanaan perkakasan kuantum
#### Cipta hirisan Circuit untuk pengundur-belakang
Ingat, fungsi ``backpropagate`` akan mengundurkan keseluruhan hirisan Circuit pada satu masa, jadi pilihan cara menghiris boleh memberi impak terhadap prestasi pengundur-belakang untuk masalah tertentu. Di sini, kita akan mengumpulkan gate yang sama jenis ke dalam hirisan menggunakan fungsi [slice_by_gate_types](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_gate_types).

Untuk perbincangan yang lebih terperinci tentang penghirisan Circuit, semak [panduan cara-cara ini](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) daripada pakej [qiskit-addon-utils](https://github.com/Qiskit/qiskit-addon-utils).

In [7]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

Below you will see that we backpropagated six slices, and the terms were combined into six and not eight groups. This implies that backpropagating one more slice would cause the number of Pauli groups to exceed eight. We can verify that this is the case by inspecting the returned metadata. Also note that in this portion the circuit transformation is exact.  That is, no terms of the new observable $O’$ were truncated. The backpropagated circuit and the backpropagated operator give the exact outcome as the original circuit and operator.

In [8]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif" alt="Output of the previous code cell" />

#### Hadkan saiz maksimum pengoperasi semasa pengundur-belakang
Semasa pengundur-belakang, bilangan terma dalam pengoperasi umumnya akan mendekati $4^N$ dengan cepat, di mana $N$ ialah bilangan Qubit. Apabila dua terma dalam pengoperasi tidak bertukar-ganti secara Qubit-wise, kita memerlukan Circuit berasingan untuk mendapatkan nilai jangkaan yang sepadan dengannya. Sebagai contoh, jika kita mempunyai observable 2-Qubit $O = 0.1 XX + 0.3 IZ - 0.5 IX$, maka oleh kerana $[XX,IX] = 0$, pengukuran dalam satu asas sudah cukup untuk mengira nilai jangkaan bagi dua terma ini. Namun, $IZ$ anti-bertukar-ganti dengan dua terma lain. Jadi kita memerlukan pengukuran asas yang berasingan untuk mengira nilai jangkaan $IZ$. Dengan kata lain, kita memerlukan dua, dan bukannya satu, Circuit untuk mengira $\langle O \rangle$. Apabila bilangan terma dalam pengoperasi bertambah, terdapat kemungkinan bilangan pelaksanaan Circuit yang diperlukan juga bertambah.

Saiz pengoperasi boleh dibatasi dengan menentukan argumen kata kunci ``operator_budget`` bagi fungsi ``backpropagate``, yang menerima instans [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget).

Untuk mengawal jumlah sumber tambahan (masa) yang diperuntukkan, kita hadkan bilangan maksimum kumpulan Pauli bertukar-ganti secara Qubit-wise yang dibenarkan untuk dimiliki oleh observable yang telah diundurkan. Di sini kita tentukan bahawa pengundur-belakang harus berhenti apabila bilangan kumpulan Pauli bertukar-ganti secara Qubit-wise dalam pengoperasi melebihi 8.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(backend)

<IBMBackend('ibm_kingston')>


In [10]:
pm_basis = generate_preset_pass_manager(
    optimization_level=3, basis_gates=backend.configuration().basis_gates
)
isa_circuit = pm_basis.run(circuit)
isa_bp_circuit = pm_basis.run(bp_circuit)

### Step 3: Execute using Qiskit primitives

First, we create two [Primitive Unified Blocs](/docs/api/qiskit/primitives) (PUBs) corresponding to the original circuit, and the backpropagated circuit. Then we execute the pubs on an ideal Estimator to obtain the expectation values.

In [11]:
pubs = [(isa_circuit, observable), (isa_bp_circuit, bp_obs)]

In [12]:
rng = np.random.default_rng()
estimator = StatevectorEstimator(seed=rng)
job = estimator.run(pubs)

### Step 4: Post-process and return result in desired classical format

Now we obtain the expectation values of the original and backpropagated circuits.

In [13]:
primitive_result = job.result()
circuit_expval = primitive_result[0].data.evs.item()
bp_circuit_expval = primitive_result[1].data.evs.item()

In [14]:
methods = [
    "No backpropagation",
    "Backpropagation",
]
values = [circuit_expval, bp_circuit_expval]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylim([0.6, 0.92])
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Perhatikan bahawa dengan memperuntukkan ralat `5e-3` setiap hirisan untuk pemangkasan, kita dapat mengeluarkan 1 hirisan lagi dari Circuit, sambil kekal dalam bajet asal lapan kumpulan Pauli bertukar-ganti dalam observable. Secara lalai, `backpropagate` menggunakan norma L1 pekali yang dipangkas untuk membatasi jumlah ralat yang ditimbulkan daripada pemangkasan. Untuk pilihan lain rujuk [panduan cara-cara tentang menentukan p_norm](https://qiskit.github.io/qiskit-addon-obp/how_tos/bound_error_using_p_norm.html).

Dalam contoh tertentu ini di mana kita telah mengundurkan tujuh hirisan, jumlah ralat pemangkasan tidak sepatutnya melebihi ``(5e-3 ralat/hirisan) * (7 hirisan) = 3.5e-2``.
Untuk perbincangan lanjut tentang pengagihan bajet ralat merentasi hirisan anda, semak [panduan cara-cara ini](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html).

In [15]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)

# Generate a time evolution circuit for the Hamiltonian
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)

# Define the observable to measure
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)

slices = slice_by_depth(circuit, max_slice_depth=1)

# Define the maximum number of qwc groups allowed in the backpropagated observable, and the truncation error budget
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

# First backpropagation without truncation
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

# Now backpropagate with truncation, using the same operator budget and the defined truncation error budget
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

# Now we transpile the original circuit and the two backpropagated circuits, and apply the layout to the corresponding observables
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = pm.run(circuit)
isa_bp_circuit = pm.run(bp_circuit)
isa_bp_circuit_trunc = pm.run(bp_circuit_trunc)

isa_observable = observable.apply_layout(isa_circuit.layout)
isa_bp_observable = bp_obs.apply_layout(isa_bp_circuit.layout)
isa_bp_observable_trunc = bp_obs_trunc.apply_layout(
    isa_bp_circuit_trunc.layout
)

# Compare the 2-qubit depth of each transpiled circuit to see how much depth backpropagation saved
print(
    f"2-qubit depth without backpropagation: {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation: {isa_bp_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation and truncation: {isa_bp_circuit_trunc.depth(lambda x: x.operation.num_qubits == 2)}"
)

pubs = [
    (isa_circuit, isa_observable),
    (isa_bp_circuit, isa_bp_observable),
    (isa_bp_circuit_trunc, isa_bp_observable_trunc),
]

# Now we instantiate the Estimator primitive for the hardware with ZNE and measurement error mitigation
# and compute the three circuits and observables
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]
estimator = EstimatorV2(mode=backend, options=options)

estimator.options.environment.job_tags = ["TUT_OBP"]
job = estimator.run(pubs)

# Retrieve the results and the standard deviations
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

2-qubit depth without backpropagation: 24
2-qubit depth with backpropagation: 20
2-qubit depth with backpropagation and truncation: 18


In [16]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.9543907942381811
Backpropagated expectation value: 0.9445337385406468
Backpropagated expectation value with truncation: 0.934050286970965


In [17]:
# Plot the results
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
error_bars = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.errorbar(methods, values, yerr=error_bars, fmt="o", color="r", capsize=5)
plt.axhline(0.89)
ax.set_ylim([0.8, 0.98])
plt.text(0.25, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/37834c72-1.avif" alt="Output of the previous code cell" />

## Next steps
If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- [Approximate quantum compilation for time evolution circuits](/docs/tutorials/approximate-quantum-compilation-for-time-evolution)
- [Multi-product formulas to reduce Trotter error](/docs/tutorials/multi-product-formula)
- [`pauli-prop`](https://github.com/Qiskit/pauli-prop), a Rust-accelerated package for Pauli propagation, with [tutorials](https://github.com/Qiskit/pauli-prop/tree/main/docs/tutorials) covering OBP, classical expectation-value estimation, and noisy simulation
</Admonition>